# SPINE-GPE v7 — PNADc Historical Backcast Publication Rerun v1.2.0

Este notebook executa o hardening corrigido com perfil de publicação. O `RUN_ID` é persistido para permitir retomada do bootstrap após interrupções.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import datetime as dt
import json
import subprocess
import sys
import pandas as pd

ROOT = Path('/content/drive/MyDrive/aCidadeAlgoritmica/SPINE-GPEv7')
SCRIPTS = ROOT / 'scripts'
SCRIPT = SCRIPTS / 'SPINE_GPEv7_PNADC_HISTORICAL_BACKCAST_HARDENING_v1.2.0.py'
REQ = SCRIPTS / 'requirements_SPINE_GPEv7_PNADC_HISTORICAL_BACKCAST_HARDENING_v1.2.0.txt'

assert SCRIPT.exists(), SCRIPT
assert REQ.exists(), REQ
print('ROOT:', ROOT)
print('SCRIPT:', SCRIPT)

## 1. Dependências

In [ ]:
install = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '-r', str(REQ)],
    text=True,
    capture_output=True,
    check=False,
)
print(install.stdout)
print(install.stderr)
print('Install exit code:', install.returncode)
assert install.returncode == 0

## 2. Run ID persistente

O mesmo ID deve ser reutilizado com `--resume-bootstrap` caso a execução seja interrompida.

In [ ]:
RUN_ID_FILE = ROOT / '00_admin' / 'PNADC_BACKCAST_PUBLICATION_RUN_ID_v120.txt'
RUN_ID_FILE.parent.mkdir(parents=True, exist_ok=True)

if RUN_ID_FILE.exists():
    RUN_ID = RUN_ID_FILE.read_text(encoding='utf-8').strip()
else:
    RUN_ID = 'publication_' + dt.datetime.now(dt.timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '_v120'
    RUN_ID_FILE.write_text(RUN_ID + chr(10), encoding='utf-8')

print('RUN_ID:', RUN_ID)
print('Arquivo de retomada:', RUN_ID_FILE)

## 3. Auditoria

Inclua em `LAYOUT_ROOTS` diretórios adicionais que contenham os layouts oficiais anuais. A lista pode permanecer vazia; nesse caso, a limitação documental será preservada como warning.

In [ ]:
LAYOUT_ROOTS = [
    # ROOT / '01_raw' / '10_ibge' / 'pnadc_layouts_anuais',
]

cmd_audit = [
    sys.executable, str(SCRIPT),
    '--root', str(ROOT),
    '--mode', 'audit',
    '--profile', 'publication',
    '--strict',
]
for layout_root in LAYOUT_ROOTS:
    cmd_audit.extend(['--layout-root', str(layout_root)])

audit = subprocess.run(cmd_audit, text=True, capture_output=True, check=False)
print(audit.stdout)
print(audit.stderr)
print('Audit exit code:', audit.returncode)
assert audit.returncode == 0

## 4. Rerun final — 500 réplicas

O checkpoint é atualizado a cada 25 réplicas. O perfil de publicação exige pelo menos 95% de réplicas bem-sucedidas.

In [ ]:
cmd_full = [
    sys.executable, str(SCRIPT),
    '--root', str(ROOT),
    '--mode', 'full',
    '--profile', 'publication',
    '--run-id', RUN_ID,
    '--bootstrap-reps', '500',
    '--bootstrap-checkpoint-every', '25',
    '--bootstrap-min-success-rate', '0.95',
    '--bootstrap-seed', '20260724',
    '--calibration-folds', '5',
    '--bootstrap-calibration-folds', '3',
    '--mca-components', '8',
    '--cluster-k-min', '4',
    '--cluster-k-max', '8',
    '--knn-min-temporal-correlation', '0.90',
    '--knn-max-relative-difference', '0.10',
    '--strict',
]
for layout_root in LAYOUT_ROOTS:
    cmd_full.extend(['--layout-root', str(layout_root)])

full = subprocess.run(cmd_full, text=True, capture_output=True, check=False)
print(full.stdout)
print(full.stderr)
print('Full exit code:', full.returncode)
assert full.returncode == 0

## 5. Retomada após interrupção

Execute esta célula somente quando a célula anterior tiver sido interrompida. Ela reutiliza o checkpoint do mesmo `RUN_ID`.

In [ ]:
cmd_resume = cmd_full + ['--resume-bootstrap']
print('Comando de retomada preparado para:', RUN_ID)
# resume = subprocess.run(cmd_resume, text=True, capture_output=True, check=False)
# print(resume.stdout)
# print(resume.stderr)
# print('Resume exit code:', resume.returncode)
# assert resume.returncode == 0

## 6. Inspeção dos artefatos

In [ ]:
TABLE = ROOT / '05_outputs' / 'tables' / 'pnadc_historical_backcast_hardening'

paths = {
    'metrics': TABLE / f'pnadc_proxy_temporal_calibrated_metrics_{RUN_ID}.csv',
    'goldens': TABLE / f'pnadc_proxy_direct_aggregate_goldens_{RUN_ID}.csv',
    'knn_aggregate': TABLE / f'pnadc_logit_knn_aggregate_stability_{RUN_ID}.csv',
    'support_summary': TABLE / f'pnadc_historical_backcast_support_summary_{RUN_ID}.csv',
    'bootstrap_summary': TABLE / f'pnadc_proxy_model_uncertainty_{RUN_ID}.csv',
    'final_estimates': TABLE / f'pnadc_historical_backcast_final_estimates_{RUN_ID}.csv',
    'mca_cells': TABLE / f'pnadc_mca_historical_cell_coordinates_{RUN_ID}.csv',
    'support': TABLE / f'pnadc_historical_transport_support_{RUN_ID}.csv',
}
for name, path in paths.items():
    print(name, '=>', path, '| exists=', path.exists())
    assert path.exists(), path

In [ ]:
metrics = pd.read_csv(paths['metrics'])
goldens = pd.read_csv(paths['goldens'])
knn_aggregate = pd.read_csv(paths['knn_aggregate'])
bootstrap_summary = pd.read_csv(paths['bootstrap_summary'])
final_estimates = pd.read_csv(paths['final_estimates'])

print('MÉTRICAS TEMPORAIS')
display(metrics)
print('GOLDEN TOTALS')
display(goldens)
print('KNN AGREGADO')
display(knn_aggregate)
print('BOOTSTRAP')
display(bootstrap_summary)
print('ESTIMATIVAS FINAIS')
display(final_estimates)

## 7. Gates finais e freeze

In [ ]:
FINAL_LOCK = ROOT / '00_admin' / 'PNADC_HISTORICAL_BACKCAST_FINAL_LOCK.json'
HARDENING_LOCK = ROOT / '00_admin' / 'PNADC_HISTORICAL_BACKCAST_HARDENING_LOCK.json'

final_lock = json.loads(FINAL_LOCK.read_text(encoding='utf-8'))
hardening_lock = json.loads(HARDENING_LOCK.read_text(encoding='utf-8'))

print(json.dumps(final_lock, ensure_ascii=False, indent=2))
assert final_lock['status'] == 'FINAL_CERTIFIED'
assert final_lock['certification_grade'] == 'PUBLICATION'
assert int(final_lock['bootstrap_reps_successful']) >= 475
assert hardening_lock['critical_failures'] == []
assert hardening_lock['run_id'] == RUN_ID

mca_hash = hardening_lock['artifact_hashes']['mca_cells']
support_hash = hardening_lock['artifact_hashes']['support']
assert mca_hash != support_hash, 'MCA e support continuam materialmente duplicados.'

print('PUBLICATION BACKCAST CERTIFIED')

## 8. Iniciar um novo rerun no futuro

Após concluir e arquivar este run, remova ou renomeie `00_admin/PNADC_BACKCAST_PUBLICATION_RUN_ID_v120.txt` para gerar um novo `RUN_ID`.